# Label-free self-supervised stereo depth

Re-implementation of the stereo architecture from *A Learned Stereo Depth System for
Robotic Manipulation in Homes* (Shankar et al., arXiv:2109.11644), trained **without any
ground-truth disparity or depth**.

## The one rule

| Mode | Ground truth |
|---|---|
| `train_unlabeled` | **never read** |
| `adapt_unlabeled` | **never read** |
| `evaluate` | read, for metrics only, on a frozen checkpoint |

The ground-truth cells below are guarded by `if MODE == "evaluate"` and will refuse to
run in a training mode.

## Enable the GPU
Settings -> Accelerator -> **GPU T4 x2** (or P100). Then Internet **on** if you want the
notebook to download datasets itself.

## 1. Mode

Set this once; every later cell reads it.

In [ ]:
MODE = "train_unlabeled"      # "train_unlabeled" | "adapt_unlabeled" | "evaluate"

# --- training ---
EPOCHS          = 30
BATCH_SIZE      = 8
LEARNING_RATE   = 1e-4
TRAIN_HEIGHT    = 384
TRAIN_WIDTH     = 640          # -> num_disparities = min(640 // 2, 384) = 320
DOWNSAMPLE      = 4            # cost volume at 1/4 resolution
TEACHER_START   = 8            # epoch the EMA teacher switches on

# --- data ---
DATASET_ROOT    = "/kaggle/working/datasets"
CUSTOM_DATA     = "/kaggle/input/my-stereo-camera"   # for adapt_unlabeled: left/ and right/

# --- evaluate ---
CHECKPOINT      = "/kaggle/working/outputs/train_unlabeled/best.pt"
PROTOCOL        = "middlebury2014"   # sceneflow | middlebury2014 | eth3d | kitti2015 | kitti2012
EVAL_ROOT       = f"{DATASET_ROOT}/middlebury/MiddEval3/trainingH"

OUTPUT_DIR      = f"/kaggle/working/outputs/{MODE}"

assert MODE in ("train_unlabeled", "adapt_unlabeled", "evaluate")
print("MODE =", MODE)

## 2. Repository and dependencies

Kaggle images already ship torch, numpy, opencv, PyYAML and matplotlib, so normally nothing is installed.

In [ ]:
import glob, os, shutil, subprocess, sys

# Set this to your own fork/clone URL if you push the repo to GitHub.
# Leave it as-is if you are attaching the code as a Kaggle Dataset instead
# (the default, and the only option that needs no GitHub account or Internet).
REPO_URL = "https://github.com/YOUR_USERNAME/stereo-depth.git"
REPO_DIR = "/kaggle/working/stereo-depth"
IS_PLACEHOLDER_URL = "YOUR_USERNAME" in REPO_URL


def _looks_like_this_repo(directory: str) -> bool:
    """A directory is treated as the source tree if it has the files this
    notebook actually imports -- not just any dataset that happens to exist."""
    return (os.path.isfile(os.path.join(directory, "train.py"))
            and os.path.isfile(os.path.join(directory, "stereo", "model", "stereo_net.py")))


def _find_attached_dataset():
    """Search every attached Kaggle input for the repo, regardless of the
    dataset's slug -- Kaggle names it from whatever title you typed, so a
    fixed name like 'stereo-depth-source' only works by coincidence."""
    if not os.path.isdir("/kaggle/input"):
        return None
    # The repo may be the dataset root itself, or one directory below it
    # (Kaggle nests an uploaded folder inside another folder of the same name).
    candidates = sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))
    for candidate in candidates:
        if os.path.isdir(candidate) and _looks_like_this_repo(candidate):
            return candidate
    return None


if not os.path.exists(REPO_DIR):
    source = _find_attached_dataset()
    if source is not None:
        print(f"found the repository attached as a Kaggle dataset: {source}")
        shutil.copytree(source, REPO_DIR)
    elif not IS_PLACEHOLDER_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        raise RuntimeError(
            "Could not find the stereo-depth source code.\n\n"
            "This notebook has no public GitHub repo configured (REPO_URL is still the "
            "placeholder), and no attached Kaggle dataset contains train.py + "
            "stereo/model/stereo_net.py. Pick one of:\n\n"
            "  A) Add Data -> Upload -> select your local stereo-depth/ folder as a new "
            "Kaggle Dataset, attach it to this notebook, then re-run this cell. Works "
            "offline and needs no GitHub account.\n\n"
            "  B) Push the repo to a GitHub repo you control and set REPO_URL above to "
            "its clone URL (https://github.com/<you>/<repo>.git), then re-run.")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Fast sanity check that the checkout is sound (~15 s).
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"], check=False)

## 3. Datasets

`python -m stereo.data.download --list` describes every option, sizes included.

**Training needs images only.** On Kaggle the practical choices are:

* small public benchmarks (Middlebury 163 MB, ETH3D 1.1 GB, KITTI 1.7 GB) — download here;
* Scene Flow (45 GB images) — attach an existing Kaggle mirror as an input dataset
  instead of downloading, and point `DATASET_ROOT` at it;
* your own capture — attach it as a Kaggle dataset with `left/` and `right/` folders.

In [ ]:
from stereo.data.download import describe, prepare, verify
print(describe())

In [ ]:
# Download the small benchmarks. Skipped in adapt mode, which uses your own data.
if MODE in ("train_unlabeled", "evaluate"):
    for name in ["middlebury"]:          # add "eth3d", "kitti2015", "kitti2012" as needed
        try:
            prepare(name, DATASET_ROOT)
        except Exception as error:
            print(f"{name} failed: {error}")

verify(DATASET_ROOT)

## 4. Look at the training pairs

Images only — no ground truth is loaded here, in any mode.

In [ ]:
import matplotlib.pyplot as plt
from stereo.data import DatasetMode, MiddleburyDataset, StereoFolderDataset
from stereo.data.augmentation import PhotometricAugmentConfig, ResizeConfig, build_train_transform

transform = build_train_transform(ResizeConfig(TRAIN_HEIGHT, TRAIN_WIDTH),
                                  PhotometricAugmentConfig(enabled=True, probability=1.0), seed=0)

if MODE == "adapt_unlabeled":
    preview = StereoFolderDataset(CUSTOM_DATA, mode=DatasetMode.TRAIN, transform=transform)
else:
    preview = MiddleburyDataset(f"{DATASET_ROOT}/middlebury/MiddEval3/trainingH",
                                mode=DatasetMode.TRAIN, transform=transform)

print(f"{len(preview)} pairs | sample keys: {sorted(preview[0])}")
assert not any(k in preview[0] for k in ("disparity_gt", "depth_gt", "valid_gt_mask"))
print("confirmed: the training sample carries no ground truth")

fig, axes = plt.subplots(3, 2, figsize=(13, 9))
for row in range(3):
    sample = preview[row % len(preview)]
    for col, view in enumerate(("left", "right")):
        axes[row][col].imshow(sample[view].permute(1, 2, 0).numpy())
        axes[row][col].set_title(f"{view} (augmented) — {sample['metadata']['sample_id']}")
        axes[row][col].axis("off")
plt.tight_layout(); plt.show()

## 5. Configuration

Built in Python so the notebook controls it directly, then written to disk so the run is reproducible from the CLI too.

In [ ]:
from stereo.config import Config, config_to_yaml
from stereo.data.augmentation import GeometricAugmentConfig
from stereo.data.registry import DatasetSpec
from stereo.model import StereoNetConfig

config = Config()
config.model = StereoNetConfig.for_width(TRAIN_WIDTH, downsample=DOWNSAMPLE)
config.dynamic_disparity = False          # already resolved above
config.data.resize = ResizeConfig(TRAIN_HEIGHT, TRAIN_WIDTH)
config.data.geometric_augmentation = GeometricAugmentConfig(enabled=True, scale=(0.8, 1.2),
                                                            aspect=(0.9, 1.1))
config.data.photometric_augmentation = PhotometricAugmentConfig(enabled=True)

if MODE == "train_unlabeled":
    # Weights are per-draw probabilities, so a 15-scene set is not drowned by a 20k one.
    config.data.train = [
        DatasetSpec(type="middlebury", root=f"{DATASET_ROOT}/middlebury/MiddEval3/trainingH",
                    weight=1.0),
        # DatasetSpec(type="sceneflow", root=f"{DATASET_ROOT}/sceneflow", weight=0.45,
        #             options={"split": "TRAIN"}),
        # DatasetSpec(type="kitti", root=f"{DATASET_ROOT}/kitti2015/training", weight=0.15,
        #             options={"version": "2015", "reference_frames_only": False}),
    ]
    config.teacher.start_epoch = TEACHER_START
    config.optimizer.learning_rate = LEARNING_RATE
else:
    config.data.train = [DatasetSpec(type="folder", root=CUSTOM_DATA, weight=1.0)]
    config.teacher.start_epoch = 2
    config.optimizer.learning_rate = LEARNING_RATE * 0.1      # gentler for domain adaptation

config.data.validation = list(config.data.train)              # label-free validation
config.training.epochs = EPOCHS
config.training.batch_size = BATCH_SIZE
config.training.num_workers = 2
config.training.use_amp = torch.cuda.is_available()
config.training.output_dir = OUTPUT_DIR
config.training.selection_metric = "val/photometric"          # LABEL-FREE selection

os.makedirs(OUTPUT_DIR, exist_ok=True)
open(f"{OUTPUT_DIR}/config.yaml", "w").write(config_to_yaml(config))
print(f"num_disparities = {config.model.num_disparities} "
      f"(min({TRAIN_WIDTH} // 2, 384), floored to a multiple of {DOWNSAMPLE})")

## 6. Build the model

Random initialisation — no ImageNet weights, no pretrained stereo weights, no pretrained Monodepth weights.

In [ ]:
from stereo.model import StereoNet

model = StereoNet(config.model)
print(f"parameters      : {model.num_parameters():,}")
print(f"cost volume     : {model.num_disparities_small} levels at 1/{model.scale}")
print(f"search range    : 0 .. {model.max_disparity} px at full resolution")
print(f"size divisor    : {model.size_divisor} (inputs are padded right/bottom, then cropped back)")

# Arbitrary resolution, including sizes that need padding.
model.eval()
for height, width in [(384, 640), (375, 1242), (540, 960), (224, 224)]:
    with torch.no_grad():
        out = model.forward_left(torch.rand(1, 3, height, width), torch.rand(1, 3, height, width))
    print(f"  {width}x{height} -> disparity {tuple(out['disparity'].shape)}")

## 7. Train, label-free

Stage 1 is photometric + smoothness + left–right consistency from random init.
Stage 2 adds the EMA teacher at epoch `TEACHER_START`, ramped in.

Watch `pseudo_cov` once the teacher starts: near 0 means the reliability filter is
rejecting everything, near 1 means it is copying the teacher's errors wholesale. The
trainer warns in both cases.

In [ ]:
if MODE == "evaluate":
    print("MODE is 'evaluate'; skipping training.")
else:
    from stereo.training import Trainer

    trainer = Trainer(config)
    best_checkpoint = trainer.fit()
    print("best label-free checkpoint:", best_checkpoint)

## 8. Label-free validation curves

No ground-truth metric is plotted — these are the quantities that actually selected the checkpoint.

In [ ]:
import json

history_path = f"{OUTPUT_DIR}/history.json"
if os.path.exists(history_path):
    history = json.load(open(history_path))
    panels = [("train/total", "total loss"), ("train/photometric", "photometric"),
              ("train/left_right", "left-right consistency"), ("train/smoothness", "smoothness"),
              ("train/pseudo_valid_ratio", "pseudo-label coverage"),
              ("train/mean_confidence", "mean confidence")]
    fig, axes = plt.subplots(2, 3, figsize=(16, 7))
    for axis, (key, title) in zip(axes.flat, panels):
        values = [record.get(key) for record in history]
        if any(v is not None for v in values):
            axis.plot([r["epoch"] for r in history], values, marker="o", ms=3)
        if key == "train/photometric" and "val/photometric" in history[0]:
            axis.plot([r["epoch"] for r in history], [r["val/photometric"] for r in history],
                      marker="s", ms=3, label="validation")
            axis.legend()
        axis.set_title(title); axis.set_xlabel("epoch"); axis.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("no history yet")

## 9. Qualitative check — still no ground truth

Predicted disparity, confidence, and the photometric reconstruction the model was actually trained on.

In [ ]:
from stereo.data import collate_samples
from stereo.geometry import warp_right_to_left
from stereo.utils.visualization import colorize, to_numpy_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

batch = collate_samples([preview[i] for i in range(min(2, len(preview)))])
left, right = batch["left_clean"].to(device), batch["right_clean"].to(device)
with torch.no_grad():
    outputs = model(left, right, directions=("left", "right"))

reconstruction, valid = warp_right_to_left(right, outputs["left"]["disparity"])
residual = (reconstruction - left).abs().mean(dim=1, keepdim=True) * valid

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for row in range(left.shape[0]):
    for axis, (image, title) in zip(axes[row], [
            (to_numpy_image(left[row:row+1]), "left"),
            (colorize(outputs["left"]["disparity"][row]), "predicted disparity"),
            (colorize(outputs["left"]["confidence"][row], 0, 1, cmap="viridis"),
             "confidence = exp(matchability)")]):
        axis.imshow(image); axis.set_title(title); axis.axis("off")
plt.tight_layout(); plt.show()
print(f"photometric residual (the training signal): {float(residual.sum() / valid.sum()):.4f}")

---
# 10. GROUND-TRUTH EVALUATION

**Everything below reads ground truth and runs only when `MODE == "evaluate"`.**

The checkpoint is loaded, frozen, and measured. No optimiser is constructed, and the
checkpoint that is being measured was selected by a label-free criterion, so ground truth
did not influence which weights got here either.

In [ ]:
if MODE != "evaluate":
    print(f"MODE is '{MODE}'. Ground truth is NOT loaded.")
    print("Set MODE = 'evaluate' and re-run from the top to benchmark a checkpoint.")
else:
    from stereo.evaluation import (PUBLISHED_RESULTS, evaluate_checkpoint, format_summary,
                                   get_protocol, write_results)
    from stereo.postprocess import PostProcessConfig
    from stereo.utils.checkpoint import build_model_from_checkpoint, checkpoint_hash

    frozen = build_model_from_checkpoint(CHECKPOINT, map_location=device)
    protocol = get_protocol(PROTOCOL)
    print(f"checkpoint : {CHECKPOINT}")
    print(f"sha256     : {checkpoint_hash(CHECKPOINT)[:16]}...")
    print(f"protocol   : {protocol.name}")
    print(f"notes      : {protocol.notes}")

### 10a. The paper's protocol: raw output

The paper's tables use *only the raw output of the learned model*, so post-processing is off.

In [ ]:
if MODE == "evaluate":
    summary_raw = evaluate_checkpoint(frozen, protocol, EVAL_ROOT, device,
                                      PostProcessConfig(enabled=False))
    summary_raw["checkpoint"] = CHECKPOINT
    write_results(summary_raw, f"{OUTPUT_DIR}/evaluation/{PROTOCOL}/raw")
    print(format_summary(summary_raw))

### 10b. With matchability post-processing

`confidence = exp(matchability) >= 0.25` and a minimum depth-region of 2000 px, as in the paper's on-robot pipeline. Reported separately, with the pixel coverage stated, so the effect of the model and the effect of the filter stay distinguishable.

In [ ]:
if MODE == "evaluate":
    summary_pp = evaluate_checkpoint(
        frozen, protocol, EVAL_ROOT, device,
        PostProcessConfig(enabled=True, confidence_threshold=0.25, min_region_pixels=2000))
    summary_pp["checkpoint"] = CHECKPOINT
    write_results(summary_pp, f"{OUTPUT_DIR}/evaluation/{PROTOCOL}/postprocessed")
    print(format_summary(summary_pp))

### 10c. Comparison table

Every row is labelled *published*, *measured* or *not available*. Nothing is invented, and rows measured under different protocols are never presented as equivalent.

In [ ]:
if MODE == "evaluate":
    def get(summary, key, variant="all"):
        # .get on the variant too: a protocol without a nonocc mask has no "nonocc" row.
        value = summary["disparity_metrics"].get(variant, {}).get(key)
        return f"{value:.3f}" if isinstance(value, (int, float)) else "n/a"

    rows = []
    if PROTOCOL == "sceneflow":
        published = PUBLISHED_RESULTS["sceneflow"]["metrics"]
        rows.append(("Original paper (Table IV)", "supervised, GT disparity",
                     f"{published['global_epe']:.3f}", f"{published['global_bad_1']:.1f}",
                     "PUBLISHED"))
        rows.append(("Reference mmstereo checkpoint", "supervised", "n/a", "n/a",
                     "NOT AVAILABLE (no released weights)"))
        rows.append(("This implementation (raw)", "label-free self-supervised",
                     get(summary_raw, "global_epe"), get(summary_raw, "global_bad_1"), "MEASURED"))
        rows.append(("This implementation (post-processed)", "label-free self-supervised",
                     get(summary_pp, "global_epe"), get(summary_pp, "global_bad_1"),
                     f"MEASURED on {summary_pp['ground_truth_pixel_coverage'] * 100:.0f}% of pixels"))
        header = ("Model", "Training", "EPE (px)", "%Bad(1.0)", "Status")
    else:
        published = PUBLISHED_RESULTS["middlebury2014_test"]["metrics"]
        rows.append(("Original paper (Table V, TEST split)", "supervised, GT disparity",
                     f"{published['image_bad_2_nonocc']}/{published['image_bad_2_all']}",
                     f"{published['image_avgerr_nonocc']}/{published['image_avgerr_all']}",
                     "PUBLISHED — DIFFERENT SPLIT"))
        rows.append(("This implementation (raw, TRAINING split)", "label-free self-supervised",
                     f"{get(summary_raw, 'image_bad_2', 'nonocc')}/{get(summary_raw, 'image_bad_2')}",
                     f"{get(summary_raw, 'image_avgerr', 'nonocc')}/{get(summary_raw, 'image_avgerr')}",
                     "MEASURED"))
        header = ("Model", "Training", "bad2.0 nocc/all", "avgerr nocc/all", "Status")

    widths = [max(len(str(row[i])) for row in [header] + rows) for i in range(len(header))]
    line = lambda row: "  ".join(str(cell).ljust(widths[i]) for i, cell in enumerate(row))
    print(f"Dataset  : {protocol.dataset_type}   Split: {protocol.split}")
    print(f"Protocol : {protocol.primary_source}")
    print(f"Mask     : {protocol.max_disparity_source}")
    print(f"Scaling  : {'median' if protocol.median_scaling else 'none (metric prediction)'}\n")
    print(line(header)); print("-" * (sum(widths) + 2 * len(widths)))
    for row in rows: print(line(row))
    if PROTOCOL == "middlebury2014":
        print(f"\nCAVEAT: {PUBLISHED_RESULTS['middlebury2014_test']['caveat']}")

### 10d. Error visualisation

Sparse ground truth is never densified for display — invalid pixels stay grey.

In [ ]:
if MODE == "evaluate":
    from stereo.data import DatasetSpec, build_benchmark_dataset
    from stereo.evaluation.disparity_metrics import disparity_valid_mask
    from stereo.postprocess import upsample_confidence

    dataset = build_benchmark_dataset(DatasetSpec(type=protocol.dataset_type, root=EVAL_ROOT,
                                                  options=dict(protocol.dataset_options)))
    for index in range(min(2, len(dataset))):
        sample = collate_samples([dataset[index]])
        left_image, right_image = sample["left"].to(device), sample["right"].to(device)
        with torch.no_grad():
            output = frozen.forward_left(left_image, right_image)

        disparity = output["disparity"]
        disparity_gt = sample["disparity_gt"].to(device)
        valid = disparity_valid_mask(disparity_gt, protocol.max_disparity, protocol.min_disparity,
                                     sample["valid_gt_mask"].to(device))
        error = (disparity - disparity_gt).abs()
        vmax = float(disparity_gt[valid].max()) if valid.any() else None

        panels = [(to_numpy_image(left_image), "left image"),
                  (colorize(disparity, 0, vmax), "predicted disparity"),
                  (colorize(disparity_gt, 0, vmax, mask=valid), "ground-truth disparity"),
                  (colorize(error, 0, 5, mask=valid, cmap="inferno"), "|error| (px), 0-5"),
                  (colorize(valid.float(), 0, 1, cmap="gray"), "valid GT mask"),
                  (colorize(upsample_confidence(output["confidence"], disparity.shape[-2:]), 0, 1,
                            cmap="viridis"), "confidence")]
        fig, axes = plt.subplots(2, 3, figsize=(17, 7))
        for axis, (image, title) in zip(axes.flat, panels):
            axis.imshow(image); axis.set_title(title, fontsize=10); axis.axis("off")
        epe = float(error[valid].mean()) if valid.any() else float("nan")
        fig.suptitle(f"{sample['metadata']['sample_id'][0]} — EPE {epe:.3f} px")
        plt.tight_layout(); plt.show()

## 11. Label-leakage audit

Runs in every mode. If this fails, nothing else in the notebook means anything.

In [ ]:
subprocess.run([sys.executable, "scripts/audit_label_leakage.py", "--strict"], check=True)

## 12. Save artefacts

Kaggle keeps `/kaggle/working`. Checkpoints and evaluation output are already there;
this just lists what a rerun would pick up.

In [ ]:
for directory, _, filenames in os.walk("/kaggle/working/outputs"):
    for filename in sorted(filenames):
        path = os.path.join(directory, filename)
        print(f"{os.path.getsize(path) / 1e6:8.2f} MB  {path}")